# Tahoe LFC No-Tahoe Embedding Results

This notebook summarizes KNN and Lasso performance for the no-Tahoe LPM coverage run. Lower `L2` is better. The fold-only tables average across cell lines within each fold before computing descriptive statistics.


In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import stats


In [2]:
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "results" / "metadata" / "fig_index.json").exists()
)
score_path = repo_root / "results" / "scores" / "no_tahoe_embeddings" / "tahoe_lfc_restricted.csv"
coverage_path = repo_root / "no_tahoe_embeddings" / "tahoe_sci_op3_no_tahoe_embeddings_coverage.csv"
df = pd.read_csv(score_path, index_col=0)
coverage = pd.read_csv(coverage_path)
print(score_path)
print(df.shape)
print(df["name"].nunique(), "models")
print(df["fold"].nunique(), "cell-line/fold units")
display(coverage)


/Users/matthewmella/repos/foundation-models-perturbation/results/scores/no_tahoe_embeddings/tahoe_lfc_restricted.csv
(12375, 11)
55 models
225 cell-line/fold units


,dataset,rows,rows_with_embedding,unique_cids,unique_cids_with_embedding
0,op3,139,136,139,136
1,sciplex3,190,177,188,175
2,tahoe,380,288,379,287


In [3]:
with open(repo_root / "results" / "metadata" / "fig_index.json", "r") as f:
    fig_index = json.load(f)

def parse_model(name):
    name = str(name)
    if name.startswith("knn_baseline_"):
        return "knn", name[len("knn_baseline_"):]
    if name.startswith("lasso_baseline_"):
        return "lasso", name[len("lasso_baseline_"):]
    return None, name

def method_map(estimator):
    mapping = {
        key.replace("HEAD_TYPE", estimator): value
        for key, value in fig_index["tahoe_lfc"].items()
    }
    mapping.update({
        f"{estimator}_baseline_LPM_emb": ["LPM", "Response Embedding"],
        f"{estimator}_baseline_ECFP:2_pkl": ["ECFP:2 (pkl)", "Molecule Structure"],
    })
    return mapping

def prepare(df, estimator):
    mapping = method_map(estimator)
    out = df.copy()
    out[["estimator", "embedding_key"]] = out["name"].apply(lambda x: pd.Series(parse_model(x)))
    out = out[out["estimator"].eq(estimator) & out["name"].isin(mapping)].copy()
    out["embedding"] = out["name"].map(lambda x: mapping[x][0])
    out["model_type"] = out["name"].map(lambda x: mapping[x][1])
    out["fold_id"] = out["fold"].astype(str).str.rsplit(".", n=1).str[-1]
    return out

def summarize_values(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    n = len(values)
    sd = values.std(ddof=1) if n > 1 else np.nan
    sem = sd / math.sqrt(n) if n > 1 else np.nan
    ci = stats.t.ppf(0.975, n - 1) * sem if n > 1 else np.nan
    return pd.Series({
        "n": n,
        "mean_l2": values.mean() if n else np.nan,
        "median_l2": np.median(values) if n else np.nan,
        "sd": sd,
        "sem": sem,
        "ci95_half_width": ci,
    })

def summarize(df, estimator):
    out = prepare(df, estimator)
    table = (
        out.groupby(["name", "embedding", "model_type"])["L2"]
        .apply(lambda s: summarize_values(s.to_numpy()))
        .unstack()
        .reset_index()
        .sort_values("mean_l2")
        .reset_index(drop=True)
    )
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    return table

def summarize_fold_only(df, estimator):
    out = prepare(df, estimator)
    fold_means = out.groupby(["name", "embedding", "model_type", "fold_id"], as_index=False)["L2"].mean()
    table = (
        fold_means.groupby(["name", "embedding", "model_type"])["L2"]
        .apply(lambda s: summarize_values(s.to_numpy()))
        .unstack()
        .reset_index()
        .rename(columns={"n": "n_folds"})
        .sort_values("mean_l2")
        .reset_index(drop=True)
    )
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    return table

def show(table, n=30):
    cols = [c for c in ["rank", "embedding", "model_type", "n", "n_folds", "mean_l2", "median_l2", "sd", "sem", "ci95_half_width"] if c in table.columns]
    display(table[cols].head(n).style.format({
        "mean_l2": "{:.3f}", "median_l2": "{:.3f}", "sd": "{:.3f}", "sem": "{:.3f}", "ci95_half_width": "{:.3f}"
    }))


In [4]:
knn = summarize(df, "knn")
knn_fold = summarize_fold_only(df, "knn")
display(Markdown("## KNN: cell-line/fold variation"))
show(knn)
display(Markdown("## KNN: fold-only variation"))
show(knn_fold)


## KNN: cell-line/fold variation

,rank,embedding,model_type,n,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,Idealized Baseline,Positive Control,225.000000,4.234,4.211,0.415,0.028,0.055
1,2,MiniMol,SMILES Transformer,225.000000,5.885,5.863,0.510,0.034,0.067
2,3,AIDO.Cell 100M - Norman,Gene Target,225.000000,5.887,5.887,0.526,0.035,0.069
3,4,Boltz Binding Probability (Protein),Protein Affinity,225.000000,5.901,5.900,0.521,0.035,0.068
4,5,MACCS,Molecule Structure,225.000000,5.903,5.910,0.520,0.035,0.068
5,6,"Targets Weighted (Name, PubChem)",Gene Target,225.000000,5.904,5.916,0.529,0.035,0.070
6,7,scPRINT - Norman,Gene Target,225.000000,5.904,5.901,0.522,0.035,0.069
7,8,Targets Weighted (Name),Gene Target,225.000000,5.905,5.920,0.532,0.035,0.070
8,9,Targets Binary (Name),Gene Target,225.000000,5.909,5.923,0.532,0.035,0.070
9,10,ErG,Molecule Structure,225.000000,5.912,5.934,0.520,0.035,0.068


## KNN: fold-only variation

,rank,embedding,model_type,n_folds,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,Idealized Baseline,Positive Control,5.000000,4.234,4.293,0.267,0.119,0.331
1,2,MiniMol,SMILES Transformer,5.000000,5.885,5.988,0.303,0.135,0.376
2,3,AIDO.Cell 100M - Norman,Gene Target,5.000000,5.887,6.022,0.337,0.151,0.418
3,4,Boltz Binding Probability (Protein),Protein Affinity,5.000000,5.901,6.024,0.330,0.147,0.409
4,5,MACCS,Molecule Structure,5.000000,5.903,6.006,0.321,0.144,0.399
5,6,"Targets Weighted (Name, PubChem)",Gene Target,5.000000,5.904,6.028,0.338,0.151,0.419
6,7,scPRINT - Norman,Gene Target,5.000000,5.904,6.029,0.327,0.146,0.406
7,8,Targets Weighted (Name),Gene Target,5.000000,5.905,6.017,0.348,0.155,0.432
8,9,Targets Binary (Name),Gene Target,5.000000,5.909,6.026,0.344,0.154,0.427
9,10,ErG,Molecule Structure,5.000000,5.912,6.019,0.324,0.145,0.402


In [5]:
lasso = summarize(df, "lasso")
lasso_fold = summarize_fold_only(df, "lasso")
display(Markdown("## Lasso: cell-line/fold variation"))
show(lasso)
display(Markdown("## Lasso: fold-only variation"))
show(lasso_fold)


## Lasso: cell-line/fold variation

,rank,embedding,model_type,n,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,Idealized Baseline,Positive Control,225.000000,2.812,2.741,0.436,0.029,0.057
1,2,ChatGPT,LLM,225.000000,5.878,5.881,0.518,0.035,0.068
2,3,MiniMol,SMILES Transformer,225.000000,5.905,5.881,0.514,0.034,0.067
3,4,LPM,Response Embedding,225.000000,5.909,5.896,0.516,0.034,0.068
4,5,Boltz (Protein),Protein Affinity,225.000000,5.911,5.894,0.518,0.035,0.068
5,6,Boltz Binding Probability (Protein),Protein Affinity,225.000000,5.912,5.903,0.519,0.035,0.068
6,7,Boltz (Fragment),Protein Affinity,225.000000,5.912,5.892,0.517,0.034,0.068
7,8,AIDO.Cell 100M - Norman,Gene Target,225.000000,5.914,5.929,0.516,0.034,0.068
8,9,ECFP:2,Molecule Structure,225.000000,5.914,5.930,0.514,0.034,0.068
9,10,Topological,Molecule Structure,225.000000,5.915,5.917,0.513,0.034,0.067


## Lasso: fold-only variation

,rank,embedding,model_type,n_folds,mean_l2,median_l2,sd,sem,ci95_half_width
0,1,Idealized Baseline,Positive Control,5.000000,2.812,2.781,0.208,0.093,0.258
1,2,ChatGPT,LLM,5.000000,5.878,5.992,0.325,0.145,0.404
2,3,MiniMol,SMILES Transformer,5.000000,5.905,6.014,0.311,0.139,0.386
3,4,LPM,Response Embedding,5.000000,5.909,6.020,0.317,0.142,0.394
4,5,Boltz (Protein),Protein Affinity,5.000000,5.911,6.022,0.318,0.142,0.395
5,6,Boltz Binding Probability (Protein),Protein Affinity,5.000000,5.912,6.022,0.320,0.143,0.397
6,7,Boltz (Fragment),Protein Affinity,5.000000,5.912,6.022,0.316,0.141,0.392
7,8,AIDO.Cell 100M - Norman,Gene Target,5.000000,5.914,6.013,0.314,0.140,0.390
8,9,ECFP:2,Molecule Structure,5.000000,5.914,6.014,0.315,0.141,0.391
9,10,Topological,Molecule Structure,5.000000,5.915,6.024,0.311,0.139,0.386


In [6]:
comparison = (
    df[df["name"].str.startswith(("knn_baseline_", "lasso_baseline_"))]
    .assign(parsed=lambda x: x["name"].apply(parse_model))
)
comparison[["estimator", "embedding_key"]] = pd.DataFrame(comparison["parsed"].tolist(), index=comparison.index)
head_compare = comparison.groupby(["embedding_key", "estimator"])["L2"].mean().unstack().dropna()
head_compare["best_head"] = np.where(head_compare["knn"] <= head_compare["lasso"], "knn", "lasso")
head_compare["lasso_minus_knn"] = head_compare["lasso"] - head_compare["knn"]
display(Markdown("## KNN vs Lasso"))
display(head_compare.sort_values("lasso_minus_knn").style.format({"knn":"{:.3f}", "lasso":"{:.3f}", "lasso_minus_knn":"{:.3f}"}))


## KNN vs Lasso

estimator,knn,lasso,best_head,lasso_minus_knn
embedding_key,,,,
pca,4.234,2.812,lasso,-1.423
ChemBERTa-77M-MLM,5.989,5.919,lasso,-0.070
chatgpt,5.941,5.878,lasso,-0.063
MolT5,5.952,5.920,lasso,-0.032
random,5.943,5.921,lasso,-0.022
unimol2-570m-H,5.944,5.923,lasso,-0.022
LPM_emb,5.930,5.909,lasso,-0.020
ChemBERTa-77M-MTR,5.933,5.921,lasso,-0.012
boltz_affinity_pred_value_fragment,5.922,5.912,lasso,-0.009


In [7]:
report_path = repo_root / "results" / "scores" / "no_tahoe_embeddings" / "tahoe_lfc_no_tahoe_embedding_report.md"

def markdown_table(frame):
    out = frame.copy()
    cols = list(out.columns)
    rows = ["| " + " | ".join(cols) + " |"]
    rows.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in out.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in cols) + " |")
    return "\n".join(rows)

def md_table(table, count_col):
    cols = ["rank", "embedding", "model_type", count_col, "mean_l2", "median_l2", "sd", "sem", "ci95_half_width"]
    out = table[cols].copy()
    for col in ["mean_l2", "median_l2", "sd", "sem", "ci95_half_width"]:
        out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
    return markdown_table(out)

parts = [
    "# Tahoe LFC No-Tahoe Embedding Report",
    "Lower `L2` is better. Cell-line/fold tables use every cell-line/fold unit; fold-only tables average cell lines within fold first.",
    "## Coverage",
    markdown_table(coverage),
    "## KNN - cell-line/fold variation",
    md_table(knn, "n"),
    "## KNN - fold-only variation",
    md_table(knn_fold, "n_folds"),
    "## Lasso - cell-line/fold variation",
    md_table(lasso, "n"),
    "## Lasso - fold-only variation",
    md_table(lasso_fold, "n_folds"),
]
report_path.write_text("\n\n".join(parts) + "\n")
report_path


PosixPath('/Users/matthewmella/repos/foundation-models-perturbation/results/scores/no_tahoe_embeddings/tahoe_lfc_no_tahoe_embedding_report.md')